# Lesson 3: Dynamic Scraping with Playwright

**Week 5 · Data Engineering Course**

---

At the end of Lesson 2 you hit a wall: `requests` fetched `quotes.toscrape.com/js/` and found **zero** quotes, even though the page is full of them in a browser. The quotes are written by JavaScript, and `requests` does not run JavaScript.

**Playwright** gets you past that wall. It drives a real browser engine — the same one Chrome uses — under your Python code's control. The browser loads the page, runs its scripts, and then you read the DOM it built.

This lesson teaches you:
- Why the async API is the one you use in a notebook
- Locators: how Playwright finds elements, and why it waits for them automatically
- Waiting properly, instead of sprinkling `sleep()`
- Clicking through pagination
- Handling infinite scroll
- Filling in forms and logging in
- Debugging with headed mode and screenshots
- How to spot the hidden JSON API and skip the browser altogether

**Install before starting** (two commands — the second downloads the browser itself, ~150 MB):
```
pip install playwright
playwright install chromium
```

In [ ]:
!pip install playwright

In [ ]:
# Downloads the Chromium build Playwright drives. Run once per machine.
!playwright install chromium

---

## 3.1 What Playwright Actually Does

```
requests:    your code  ──HTTP──>  server          (you get the HTML the server sent)

Playwright:  your code  ──>  real browser  ──HTTP──>  server
                                  │
                                  └─ runs JavaScript, applies CSS, fires XHRs,
                                     builds the DOM — then you read the result
```

Playwright can drive three engines: **chromium** (use this), **firefox**, and **webkit** (Safari's engine). It runs **headless** by default — the browser is a real browser but no window appears, which is what you want on a server.

### Playwright vs Selenium

Selenium is the tool you will see in older tutorials. Playwright is what a new project should use:

| | Playwright | Selenium |
|---|---|---|
| Browser drivers | downloads and manages them for you | you install and version-match ChromeDriver yourself |
| Waiting | **automatic** — every action waits for the element | mostly manual (`WebDriverWait`) |
| API | one object model, async and sync | older, more verbose |
| Network interception | built in | awkward |

### The cost of a browser

A browser is expensive. Rough numbers for one page:

| | requests + BeautifulSoup | Playwright |
|---|---|---|
| Time | 0.2–0.5 s | 1–3 s |
| Memory | a few MB | 150–400 MB per browser |
| Breaks when | HTML structure changes | HTML structure changes |

So the rule is: **use `requests` whenever it works, and reach for Playwright only when it does not.** A pipeline that scrapes 50,000 pages a night with a browser is a pipeline that needs a rethink.

---

## 3.2 Sync API or Async API? In a Notebook, Async.

Playwright ships two identical APIs.

**The sync API** is what you use in a normal `.py` script:

```python
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch()
    page = browser.new_page()
    page.goto('https://quotes.toscrape.com/js/')
    print(page.title())
    browser.close()
```

Run that in a Jupyter cell and it fails:

```
Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.
```

The reason: a notebook kernel is **already running an asyncio event loop** so it can await things for you. The sync API refuses to start a second one.

**The async API** is the notebook version. Add `async`/`await` and it works, because Jupyter lets you `await` at the top level of a cell:

```python
from playwright.async_api import async_playwright

pw      = await async_playwright().start()
browser = await pw.chromium.launch()
page    = await browser.new_page()
await page.goto('https://quotes.toscrape.com/js/')
print(await page.title())
await browser.close()
await pw.stop()
```

We use `.start()` / `.stop()` rather than `async with` so the browser stays alive **across cells** — that is the whole point of a notebook: poke at a live page one cell at a time.

> **Remember:** every Playwright call in the async API returns a coroutine. Forget one `await` and you get `<coroutine object ...>` instead of your data, or a `RuntimeWarning: coroutine was never awaited`. When something returns a coroutine object, you missed an `await`.

### Windows: if you get `NotImplementedError`

Launching a browser needs asyncio subprocess support, and some older Jupyter builds on Windows install an event loop policy that lacks it. If `pw.chromium.launch()` raises `NotImplementedError`:

1. Upgrade the kernel stack first — this is fixed in current versions:
   ```
   pip install -U jupyter ipykernel tornado
   ```
   Then restart the kernel.
2. If it persists, run your scraper as a **`.py` script with the sync API** (`python scrape_jobs.py`) instead of in the notebook. Scripts are unaffected because they own their event loop — and a production scraper should be a script anyway. Lab 5 is built that way on purpose.

In [ ]:
import re
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout

DATA  = Path('data')
CLEAN = DATA / 'clean'
SHOTS = DATA / 'screenshots'
for p in [CLEAN, SHOTS]:
    p.mkdir(parents=True, exist_ok=True)

print('Playwright imported.')

In [ ]:
# Start Playwright and launch a browser. These objects stay alive for the whole notebook.
pw = await async_playwright().start()

browser = await pw.chromium.launch(
    headless=True,        # True = no window. Set False to watch it work (see 3.9).
)

# A CONTEXT is an isolated browser profile: its own cookies, storage and settings.
# Use one context per "session" you want to keep separate.
context = await browser.new_context(
    user_agent='DE-Course-Scraper/1.0 (student project; contact: you@example.com)',
    viewport={'width': 1280, 'height': 900},
    locale='en-GB',
)

page = await context.new_page()

print(f'Browser: {browser.browser_type.name} {browser.version}')

In [ ]:
# The page that defeated requests in Lesson 2
await page.goto('https://quotes.toscrape.com/js/', wait_until='networkidle')

print('title:       ', await page.title())
print('url:         ', page.url)                      # a property, no await
print('html length: ', len(await page.content()))
print('quotes found:', await page.locator('.quote').count())     # 10, not 0

**Ten quotes.** Same URL, same selector — the difference is that a browser ran the JavaScript that created those elements.

---

## 3.3 Locators

A **locator** is not an element. It is a *description* of how to find an element, which Playwright re-resolves every time you use it. That one design choice removes most of the flakiness that plagues browser automation: the element can be re-rendered between two of your lines and the locator still works.

```python
quotes = page.locator('.quote')     # no await — building it touches nothing
n = await quotes.count()            # await — now the page is queried
```

| Call | Returns |
|------|---------|
| `page.locator(css)` | a locator for every match |
| `await loc.count()` | how many elements match |
| `loc.first` / `loc.last` / `loc.nth(i)` | a locator for one of them |
| `await loc.all()` | a list of locators, one per match |
| `await loc.inner_text()` | the rendered text (what you see) |
| `await loc.text_content()` | the raw text (includes hidden text) |
| `await loc.all_inner_texts()` | a list of strings — every match at once |
| `await loc.get_attribute('href')` | one attribute |
| `loc.locator(css)` | narrow the search **inside** that locator |

### Strictness

Any action or text read on a locator that matches **more than one** element raises an error:

```
Error: strict mode violation: locator('.quote') resolved to 10 elements
```

This is deliberate: it stops you silently reading the wrong row. Be specific (`.nth(0)`, `.first`) or loop over `all()`.

### Prefer user-facing locators for *interaction*

For clicking and typing, Playwright recommends locating things the way a user would describe them:

```python
page.get_by_role('button', name='Login')      # by accessible role + name
page.get_by_label('Username')                 # the input belonging to a label
page.get_by_text('Next')                      # by visible text
page.get_by_placeholder('Search...')
```

These survive restyling far better than CSS paths. For bulk **data extraction**, CSS selectors are usually clearer — you already know them from Lesson 1.

In [ ]:
quotes = page.locator('.quote')           # building a locator: no await needed
print('count:', await quotes.count())

first = quotes.first
print('text:  ', (await first.locator('.text').inner_text())[:70])
print('author:', await first.locator('.author').inner_text())
print('tags:  ', await first.locator('.tag').all_inner_texts())

In [ ]:
# Strict mode protects you from reading the wrong element by accident
try:
    await quotes.inner_text()        # 10 matches -> refuses
except Exception as e:
    print(f'{type(e).__name__}: {str(e).splitlines()[0]}')

print()
print('Be specific instead:')
print(' nth(2):', (await quotes.nth(2).locator('.author').inner_text()))

In [ ]:
# Extract every quote on the page by looping over locators
rows = []
for q in await quotes.all():
    rows.append({
        'text':   (await q.locator('.text').inner_text()).strip('\u201c\u201d'),
        'author': await q.locator('.author').inner_text(),
        'tags':   ', '.join(await q.locator('.tag').all_inner_texts()),
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

---

## 3.4 The Hybrid Pattern: Playwright to Render, BeautifulSoup to Parse

Looping over locators means one round-trip to the browser per field — fine for 10 quotes, slow for 500 rows, and verbose either way.

The pattern most production scrapers use instead:

```
Playwright: load the page, wait for it, handle the clicking  →  page.content()
BeautifulSoup: parse that HTML string, fast and offline
```

`await page.content()` returns the **current, fully-rendered** HTML — JavaScript already applied. Everything you learned in Lessons 1 and 2 then works unchanged, and your parsing functions stay testable against saved HTML files with no browser in sight.

Use locators for **interaction** (clicking, typing, waiting). Use BeautifulSoup for **extraction**.

In [ ]:
def parse_quotes(html):
    '''Parse quotes out of a page's HTML.

    Note what this function does NOT touch: Playwright. It takes a string
    and returns rows, so you can test it against a saved .html file with no
    browser running. Keep your parsing free of your fetching.
    '''
    soup = BeautifulSoup(html, 'html.parser')
    return [
        {
            'text':   q.select_one('.text').get_text(strip=True).strip('\u201c\u201d'),
            'author': q.select_one('.author').get_text(strip=True),
            'tags':   ', '.join(t.get_text(strip=True) for t in q.select('.tag')),
        }
        for q in soup.select('.quote')
    ]


# Playwright renders, BeautifulSoup extracts
hybrid = pd.DataFrame(parse_quotes(await page.content()))

print(f'locator version: {len(df)} rows')
print(f'hybrid version:  {len(hybrid)} rows')
hybrid.head(3)

---

## 3.5 Waiting — the Thing Everyone Gets Wrong

A browser page is not finished when `goto()` returns. Data arrives over XHR, elements render, spinners disappear. The classic beginner scraper is:

```python
await page.goto(url)
await asyncio.sleep(5)      # "give it time to load"
```

That is wrong twice: it wastes 5 seconds when the page was ready in 0.3, and it still fails on the slow day.

### What Playwright does for you

Every **action** (`click`, `fill`, `inner_text`) auto-waits for its element to exist, be visible, and be stable — up to a 30-second default timeout. Most of the time you do not write waiting code at all.

### What you still control

| Tool | When to use it |
|------|----------------|
| `await page.goto(url, wait_until='domcontentloaded')` | fastest; the HTML is parsed |
| `... wait_until='load'` | default; images and sub-resources done |
| `... wait_until='networkidle'` | no network traffic for 500 ms — good for XHR-driven pages, slow and occasionally never true on pages that poll |
| `await page.wait_for_selector('.quote', timeout=5000)` | wait for the specific thing you need. **The best default.** |
| `await loc.wait_for(state='visible')` | wait on an existing locator |
| `async with page.expect_response(pred): ...` | wait for a specific network response |
| `await page.wait_for_timeout(500)` | a real pause. A last resort, but at least it is Playwright's own and does not block the loop |
| `page.set_default_timeout(15000)` | change the 30 s default for this page |

**Wait for the thing you need, not for time to pass.** `wait_for_selector('.quote')` says exactly what "ready" means for your scraper.

A timeout raises `playwright.async_api.TimeoutError`. Catch it: on a listing page, "no results" and "page broken" both look like a missing selector, and your pipeline should tell them apart.

In [ ]:
# Wait for what you need, with a short timeout, and handle the failure
await page.goto('https://quotes.toscrape.com/js/', wait_until='domcontentloaded')

try:
    await page.wait_for_selector('.quote', timeout=5000)
    print(f'ready: {await page.locator(".quote").count()} quotes')
except PlaywrightTimeout:
    print('No .quote appeared within 5s — empty results, or the page changed.')

In [ ]:
# A timeout you can see happening: nothing on this page has class .job-card
try:
    await page.wait_for_selector('.job-card', timeout=2000)
except PlaywrightTimeout as e:
    print(f'TimeoutError after 2s, as expected')
    print(f'  {str(e).splitlines()[0]}')

---

## 3.6 Clicking Through Pagination

In Lesson 2 you followed the `next` link by fetching its URL. With a browser you can instead **click** it — which matters when the button is a JavaScript handler with no `href` to fetch.

The loop is always the same shape:

```
while there is a Next button and we are under the page cap:
    extract the current page
    click Next
    wait for the new content
```

The subtlety: after clicking, the new rows may take a moment to appear, and Playwright's auto-wait only covers the click itself. Waiting for the *selector* again is usually enough; on a page that replaces content in place, wait for something that proves the new page arrived — a changed page number, or `expect_response` on the data call.

In [ ]:
async def scrape_paginated(page, start_url, max_pages=3):
    '''Click "Next" until it disappears or we reach max_pages.'''
    await page.goto(start_url, wait_until='domcontentloaded')
    all_rows = []

    for page_no in range(1, max_pages + 1):
        await page.wait_for_selector('.quote', timeout=10000)

        rows = parse_quotes(await page.content())
        all_rows.extend(rows)
        print(f'page {page_no}: +{len(rows)} rows (total {len(all_rows)})  {page.url}')

        if page_no == max_pages:
            print(f'reached max_pages={max_pages}')
            break

        next_link = page.locator('li.next a')
        if await next_link.count() == 0:
            print('no Next link — last page')
            break

        await next_link.click()
        # Prove the new page actually arrived before extracting again
        await page.wait_for_url(re.compile(r'/page/\d+/'), timeout=10000)

    return pd.DataFrame(all_rows)


quotes_df = await scrape_paginated(page, 'https://quotes.toscrape.com/js/', max_pages=3)
print(f'\nTotal: {len(quotes_df)} quotes')
quotes_df.tail(3)

---

## 3.7 Infinite Scroll

Plenty of sites have no Next button at all: they load more rows as you scroll. `https://quotes.toscrape.com/scroll` is one.

You cannot fetch your way through that — the rows only exist once the browser has scrolled and the XHR has returned. The algorithm:

```
repeat:
    count the rows
    scroll to the bottom
    wait for the count to grow
until the count stops growing (or you hit your cap)
```

The termination condition is **the count stopped changing**, not a fixed number of scrolls. And always cap the loop: an infinite scroll with a bug really is infinite.

In [ ]:
async def scrape_infinite_scroll(page, url, max_rounds=12, row_selector='.quote'):
    '''Scroll until no new rows appear, then return the full page HTML.'''
    await page.goto(url, wait_until='domcontentloaded')
    await page.wait_for_selector(row_selector, timeout=10000)

    seen = await page.locator(row_selector).count()
    print(f'start: {seen} rows')

    for round_no in range(1, max_rounds + 1):
        # Scroll to the bottom of the document
        await page.evaluate('window.scrollTo(0, document.body.scrollHeight)')

        try:
            # Wait for the row count to actually increase — not for a fixed time
            await page.wait_for_function(
                'n => document.querySelectorAll(".quote").length > n',
                arg=seen,
                timeout=4000,
            )
        except PlaywrightTimeout:
            print(f'round {round_no}: no new rows — reached the end')
            break

        seen = await page.locator(row_selector).count()
        print(f'round {round_no}: {seen} rows')
    else:
        print(f'stopped at max_rounds={max_rounds} — there may be more')

    return await page.content()


html = await scrape_infinite_scroll(page, 'https://quotes.toscrape.com/scroll')
scrolled = pd.DataFrame(parse_quotes(html))
print(f'\nCollected {len(scrolled)} quotes by scrolling')
scrolled.head(3)

---

## 3.8 Forms, Buttons and Logging In

Some data sits behind a search form or a login. Playwright fills forms exactly as a user does, which means hidden CSRF tokens, JavaScript validation, and cookies all just work — none of which is true if you try to POST the form yourself with `requests`.

```python
await page.get_by_label('Username').fill('student')
await page.get_by_label('Password').fill('student')
await page.get_by_role('button', name='Login').click()
```

Cookies live in the **context**, so every page you open from that context stays logged in. `context.storage_state(path='auth.json')` saves that session to a file and `browser.new_context(storage_state='auth.json')` restores it — so a scheduled pipeline logs in once a week rather than once a run.

> **Only log in where you are allowed to.** A login means there are terms you agreed to. Scraping behind someone else's credentials, or circumventing a paywall, is the line between scraping and unauthorised access. `quotes.toscrape.com` is a sandbox that accepts any username and password — that is why we use it.

In [ ]:
# quotes.toscrape.com/login accepts anything. The form has a hidden CSRF token
# that we never touch — the browser submits it for us.
await page.goto('https://quotes.toscrape.com/login')

await page.fill('#username', 'student')
await page.fill('#password', 'student')
await page.click('input[type="submit"]')

await page.wait_for_selector('a[href="/logout"]', timeout=10000)
print('Logged in — the page now offers a Logout link')
print('url:', page.url)

# Cookies belong to the context, so they persist across pages
cookies = await context.cookies()
print('cookies:', [c['name'] for c in cookies])

---

## 3.9 Debugging: Watch It, Screenshot It, Slow It Down

When a scraper fails on a server at 3 a.m., you have no screen to look at. Build your own eyes:

| Technique | How |
|-----------|-----|
| **Watch the browser** | `launch(headless=False)` — a window opens and you see every step |
| **Slow it down** | `launch(headless=False, slow_mo=500)` — half a second between actions |
| **Screenshot on failure** | `await page.screenshot(path='fail.png', full_page=True)` in your `except` block |
| **Save the HTML on failure** | `Path('fail.html').write_text(await page.content())` |
| **Read the console** | `page.on('console', lambda m: print(m.text))` — catches the site's own JS errors |
| **Step through interactively** | `await page.pause()` opens Playwright Inspector (headed mode only) |
| **Generate selectors for you** | run `playwright codegen https://example.com` in a terminal: click around, and it prints the Python |

`playwright codegen` deserves emphasis. It opens a browser, records what you do, and writes working Playwright code with sensible locators. For a fiddly multi-step flow, record it, then clean up what it gives you.

**Screenshot on failure is not optional** in a scheduled scraper. "It returned 0 rows" tells you nothing; a screenshot showing a cookie banner, a captcha, or a "we've redesigned our site" notice tells you everything.

In [ ]:
# Screenshot the current page — the first thing to do when a scrape returns nothing
await page.goto('https://quotes.toscrape.com/js/', wait_until='networkidle')

shot = SHOTS / 'quotes_js.png'
await page.screenshot(path=str(shot), full_page=True)
print(f'saved {shot}  ({shot.stat().st_size / 1024:.0f} KB)')

# A scrape wrapped in its own evidence-collector
async def scrape_with_evidence(page, url, selector):
    try:
        await page.goto(url, wait_until='domcontentloaded')
        await page.wait_for_selector(selector, timeout=5000)
        return await page.locator(selector).count()
    except PlaywrightTimeout:
        await page.screenshot(path=str(SHOTS / 'failure.png'), full_page=True)
        (SHOTS / 'failure.html').write_text(await page.content(), encoding='utf-8')
        print(f'FAILED on {url}: no {selector}. Screenshot and HTML saved for inspection.')
        return 0

print('rows:', await scrape_with_evidence(page, 'https://quotes.toscrape.com/js/', '.quote'))
print('rows:', await scrape_with_evidence(page, 'https://quotes.toscrape.com/js/', '.job-card'))

---

## 3.10 The Shortcut: Find the Hidden API

Before you commit to a browser, look at what the page's JavaScript is doing. A page that renders client-side had to **get the data from somewhere** — almost always a JSON endpoint.

Find it in DevTools: **Network → Fetch/XHR**, reload, and look at the responses. Or let Playwright tell you, with `page.expect_response()` or `page.on('response', ...)`.

The infinite-scroll page we just scrolled through calls `/api/quotes?page=N` and gets clean JSON back. Once you know that, the browser is dead weight: call the endpoint with `requests` and you get typed data, 100× faster, with no selectors to break.

This is the highest-value habit in this lesson. **Check for the XHR before you write the scraper.**

In [ ]:
# Let Playwright show you every XHR the page makes
seen_requests = []
page.on('response', lambda r: seen_requests.append((r.status, r.request.resource_type, r.url)))

await page.goto('https://quotes.toscrape.com/scroll', wait_until='networkidle')

print('Requests the page made:')
for status, kind, url in seen_requests:
    if kind in ('xhr', 'fetch'):
        print(f'  {status}  {kind:<5}  {url}')

In [ ]:
# Or wait for one specific response and read its body
async with page.expect_response(lambda r: '/api/quotes' in r.url) as caught:
    await page.goto('https://quotes.toscrape.com/scroll')

response = await caught.value
payload  = await response.json()

print('status:  ', response.status)
print('url:     ', response.url)
print('keys:    ', list(payload.keys()))
print('quotes:  ', len(payload['quotes']))
print('has_next:', payload['has_next'])

In [ ]:
# ...and now throw the browser away. requests does the whole job.
import requests

all_quotes, page_no = [], 1
while True:
    r = requests.get('https://quotes.toscrape.com/api/quotes',
                     params={'page': page_no}, timeout=10)
    r.raise_for_status()
    data = r.json()

    all_quotes.extend({
        'text':   q['text'].strip('\u201c\u201d'),
        'author': q['author']['name'],
        'tags':   ', '.join(q['tags']),
    } for q in data['quotes'])

    if not data['has_next'] or page_no >= 10:
        break
    page_no += 1

api_df = pd.DataFrame(all_quotes)
print(f'{len(api_df)} quotes from {page_no} API calls — no browser, no selectors')
api_df.head(3)

In [ ]:
# Save what we collected, then shut the browser down.
# ALWAYS close: a leaked browser process holds hundreds of MB of memory.
quotes_df.to_csv(CLEAN / 'quotes_playwright.csv', index=False)
api_df.to_csv(CLEAN / 'quotes_api.csv', index=False)
print(f'Saved {len(quotes_df)} + {len(api_df)} rows to {CLEAN}')

await context.close()
await browser.close()
await pw.stop()
print('Browser closed, Playwright stopped.')

---

## 3.11 The Same Scraper as a Script

A notebook is for exploring. A pipeline is a `.py` file that cron or Airflow can run. Here is the sync-API version of everything above — and the shape Lab 5 uses:

```python
# scrape_quotes.py
import pandas as pd
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright

URL = 'https://quotes.toscrape.com/js/'


def extract(html):
    soup = BeautifulSoup(html, 'html.parser')
    return [
        {
            'text':   q.select_one('.text').get_text(strip=True).strip('\u201c\u201d'),
            'author': q.select_one('.author').get_text(strip=True),
            'tags':   ', '.join(t.get_text(strip=True) for t in q.select('.tag')),
        }
        for q in soup.select('.quote')
    ]


def scrape(max_pages=3):
    rows = []
    with sync_playwright() as pw:                 # no await anywhere
        browser = pw.chromium.launch(headless=True)
        page    = browser.new_page()
        page.goto(URL, wait_until='domcontentloaded')

        for _ in range(max_pages):
            page.wait_for_selector('.quote', timeout=10_000)
            rows += extract(page.content())

            nxt = page.locator('li.next a')
            if nxt.count() == 0:
                break
            nxt.click()

        browser.close()
    return pd.DataFrame(rows)


if __name__ == '__main__':
    df = scrape()
    df.to_csv('data/clean/quotes.csv', index=False)
    print(f'{len(df)} quotes saved')
```

Run it with `python scrape_quotes.py`. Same logic, no `await`, no event-loop problems on any platform — and it can be scheduled, tested, and imported.

---

## Key Takeaways

1. Playwright drives a **real browser engine**, so it sees the page *after* JavaScript has run. That is the only way to scrape a client-rendered site.
2. A browser costs ~10× the time and ~100× the memory of `requests`. **Use `requests` whenever it works.**
3. In a **notebook** you must use the **async API** (`from playwright.async_api import async_playwright`) because the kernel already owns an event loop. In a **`.py` script**, use the sync API. Forgetting an `await` gives you a coroutine object instead of data.
4. `.start()` / `.stop()` keeps the browser alive across notebook cells; `async with` is for a single self-contained block.
5. A **locator describes** how to find an element and re-resolves it on every use — which is why Playwright is so much less flaky than older tools. Build locators without `await`; reading from them needs `await`.
6. **Strict mode** raises an error when a locator matches several elements. Use `.first`, `.nth(i)`, or loop over `await loc.all()`.
7. Playwright **auto-waits** on every action. Beyond that, wait for **the thing you need** (`wait_for_selector`, `wait_for_function`, `expect_response`) — never for a fixed number of seconds.
8. Catch `playwright.async_api.TimeoutError` explicitly. "No rows" and "site broke" look identical otherwise.
9. Use the **hybrid pattern**: Playwright to load, click and wait; `await page.content()` + BeautifulSoup to extract. Your parsing code then has no browser dependency and can be tested on saved HTML.
10. **Infinite scroll** terminates when the row count stops growing, not after N scrolls — and always with a hard cap on rounds.
11. Cookies live in the **context**. `storage_state` saves and restores a logged-in session so a scheduled job does not log in every run. Only log in where you are permitted to.
12. **Screenshot and save the HTML on failure**, always. On a server it is the only evidence you will ever have of why a run returned zero rows.
13. `playwright codegen <url>` records your clicks and writes the Python for you — the fastest way to build a multi-step flow.
14. **Check DevTools → Network → Fetch/XHR before writing any scraper.** If the page loads its data as JSON, call that endpoint with `requests` and skip the browser entirely.
15. Always `await browser.close()`. A leaked browser process is hundreds of megabytes that never come back.

---

**Next:** Lesson 4 — CSV, JSON, Excel and Parquet: where scraped data goes and which format to choose.